### Requirements

- Be able to create a grid and crop out patches of a specified size (224x224) and save them to a specified directory.
- Costumize the patching to each class/image
- Discard patches with no plants.
- Background subtraction:
    - Remove coins
    - Remove boxes, other objects
    - Have only the plants in the patches.

In [9]:
import os
import cv2

import numpy as np

In [10]:
data_dir = '/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/preprocessed'
target_dir = '/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/cropped'
os.makedirs(target_dir, exist_ok=True)


In [11]:
# Key codes for OpenCV (on Linux)
KEY_LEFT = 65361
KEY_RIGHT = 65363
KEY_UP = 65362
KEY_DOWN = 65364
KEY_ESC = 27
DIR_KEYS = [KEY_LEFT, KEY_RIGHT, KEY_UP, KEY_DOWN]

class OpenCVGUI:
    def __init__(self, folder_path, output_path=None):
        
        # GUI parameters
        self.window_width = 800
        self.window_height = 600
        
        # Paths
        self.folder_path = folder_path
        folder_name = os.path.basename(folder_path)       
        if output_path is not None:
            self.output_folder = os.path.join(output_path, folder_name)
        else:
            self.output_folder = os.path.join(folder_path, 'patches')
        os.makedirs(self.output_folder, exist_ok=True)
        self.img_paths = []
        
        for img_filename in os.listdir(folder_path): # Renamed 'img' to 'img_filename'
            if img_filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                self.img_paths.append(os.path.join(folder_path, img_filename))
        
        cv2.namedWindow('Image', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Image', self.window_width, self.window_height)
        
        # Patching parameters
        self.base_patch_size = 224
        self.patch_size_multiplier = 1
        self.patch_vertical_offset = 0      # In original image coordinates
        self.patch_horizontal_offset = 0    
        self.patch_offset_step = 10         
        
    # def hsv_filtering(self, img_path):
        
    #     # HSV Filtering
    #     self.hsv_lower = np.array([0, 0, 0])
    #     self.hsv_upper = np.array([180, 255, 255])
        
    #     cv2.namedWindow('HSV Filter', cv2.WINDOW_NORMAL)
    #     cv2.resizeWindow('HSV Filter', self.window_width, self.window_height)
        
    #     # Trackbars for HSV filtering
    #     cv2.createTrackbar('H Lower', 'HSV Filter', 0, 180, lambda x: None)        
    #     cv2.createTrackbar('H Upper', 'HSV Filter', 180, 180, lambda x: None)
    #     cv2.createTrackbar('S Lower', 'HSV Filter', 0, 255, lambda x: None)
    #     cv2.createTrackbar('S Upper', 'HSV Filter', 255, 255, lambda x: None)
    #     cv2.createTrackbar('V Lower', 'HSV Filter', 0, 255, lambda x: None)
    #     cv2.createTrackbar('V Upper', 'HSV Filter', 255, 255, lambda x: None)
        
    #     # Files for HSV Values
    #     self.hsv_good_values_file = os.path.join(self.folder_path, './config/hsv_good_values.npy')
    #     self.hsv_bad_values_file = os.path.join(self.folder_path, './config/hsv_bad_values.npy')
    #     os.makedirs(os.path.dirname(self.hsv_good_values_file), exist_ok=True)
    #     os.makedirs(os.path.dirname(self.hsv_bad_values_file), exist_ok=True)
        
        
    #     def update_hsv_values(self):
    #         self.hsv_lower = np.array([
    #             cv2.getTrackbarPos('H Lower', 'HSV Filter'),
    #             cv2.getTrackbarPos('S Lower', 'HSV Filter'),
    #             cv2.getTrackbarPos('V Lower', 'HSV Filter')
    #         ])
            
    #         self.hsv_upper = np.array([
    #             cv2.getTrackbarPos('H Upper', 'HSV Filter'),
    #             cv2.getTrackbarPos('S Upper', 'HSV Filter'),
    #             cv2.getTrackbarPos('V Upper', 'HSV Filter')
    #         ])
        
    #     img = cv2.imread(img_path)
    #     img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
    #     while True:
    #         update_hsv_values(self)
            
    #         mask = cv2.inRange(img_hsv, self.hsv_lower, self.hsv_upper)
    #         filtered_img = cv2.bitwise_and(img, img, mask=mask)
            
    #         cv2.imshow('HSV Filter', filtered_img)
            
    #         key = cv2.waitKeyEx(1)
            
    #         if key == ord('g'):
    #             np.save(self.hsv_good_values_file, self.hsv_lower)
            
    #         elif key == ord('b'):
    #             np.save(self.hsv_bad_values_file, self.hsv_upper)
                
    #         elif key == ord('q') or key == KEY_ESC:
    #             cv2.destroyWindow('HSV Filter')
    #             return
           
    def draw_grid(self, img_to_draw_on, scale=1, thickness=2):
        # img_to_draw_on is the resized image.
        # scale is the visualization_scale (resized_dim / original_dim).
        
        height_resized, width_resized = img_to_draw_on.shape[:2]
        
        # Calculate patch size as it appears on the resized image
        scaled_patch_size = int(self.base_patch_size * self.patch_size_multiplier * scale)        
        if scaled_patch_size <= 0:
            scaled_patch_size = 1 

        # Offsets are in original image coordinates. Scale them for the resized image.
        scaled_vertical_offset = int(self.patch_vertical_offset * scale)
        scaled_horizontal_offset = int(self.patch_horizontal_offset * scale)

        for y_coord in range(scaled_vertical_offset, height_resized, scaled_patch_size):
            cv2.line(img_to_draw_on, (0, y_coord), (width_resized, y_coord), (255, 0, 0), thickness)
        
        for x_coord in range(scaled_horizontal_offset, width_resized, scaled_patch_size):
            cv2.line(img_to_draw_on, (x_coord, 0), (x_coord, height_resized), (255, 0, 0), thickness)      

        return img_to_draw_on

    def resize_to_fit(self, img):
        img_h, img_w = img.shape[:2]
        if img_w == 0 or img_h == 0: 
            return cv2.resize(img, (1,1), interpolation=cv2.INTER_NEAREST), 1.0

        scale = min(self.window_width / img_w, self.window_height / img_h)
        if scale <= 0: 
             scale = 1.0 

        new_width = int(img_w * scale)
        new_height = int(img_h * scale)
        
        new_width = max(1, new_width)
        new_height = max(1, new_height)
        
        return cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA), scale
               
    def run(self):
        for img_path in self.img_paths:
            print(f"Processing image: {img_path}")
            
            original_img = cv2.imread(img_path)
            if original_img is None:
                print(f"Error loading image: {img_path}")
                continue
            
            print(f"Image shape: {original_img.shape}")

            # Note: offsets (self.patch_vertical_offset, etc.) persist between images unless reset.
            # To reset for each image, uncomment below:
            # self.patch_vertical_offset = 0
            # self.patch_horizontal_offset = 0
            # self.patch_size_multiplier = 1

            img_resized, visualization_scale = self.resize_to_fit(original_img)
            
            # Initial display
            img_grided = self.draw_grid(img_resized.copy(), scale=visualization_scale) # Draw on a copy
            cv2.imshow('Image', img_grided)
            
            # Event loop for the current image
            while True:
                key = cv2.waitKeyEx(0) 

                action_taken = False # Flag to check if redraw is needed
                
                # if key == ord('h'):
                #     print("Entering HSV Filter mode.")
                #     self.hsv_filtering(img_path)
                #     key = cv2.waitKeyEx(0)
                
                if key == ord('m'):
                    print("Increasing patch size multiplier.")
                    self.patch_size_multiplier += 1
                    action_taken = True
                elif key == ord('n'):
                    if self.patch_size_multiplier > 1:
                        print("Decreasing patch size multiplier.")
                        self.patch_size_multiplier -= 1
                        action_taken = True
                    else:
                        print("Patch size multiplier is already at minimum (1).")
                elif key in DIR_KEYS:
                    action_taken = True
                    if key == KEY_LEFT:
                        print("Moving grid left.")
                        self.patch_horizontal_offset -= self.patch_offset_step
                    elif key == KEY_RIGHT:
                        print("Moving grid right.")
                        self.patch_horizontal_offset += self.patch_offset_step
                    elif key == KEY_UP:
                        print("Moving grid up.")
                        self.patch_vertical_offset -= self.patch_offset_step
                    elif key == KEY_DOWN:
                        print("Moving grid down.")
                        self.patch_vertical_offset += self.patch_offset_step
                    print(f"Current offsets - Vertical: {self.patch_vertical_offset}, Horizontal: {self.patch_horizontal_offset}")
                    
                elif key == ord('s'):
                    self.save_patches(img_path) # Uses original_img via img_path
                    continue 
                
                elif key == ord('c'): 
                    print("Continuing to next image.")
                    break 
                
                elif key == ord('q') or key == KEY_ESC: 
                    print("Quitting.")
                    cv2.destroyAllWindows()
                    return 
                
                if action_taken:
                    img_grided = self.draw_grid(img_resized.copy(), scale=visualization_scale)
                    cv2.imshow('Image', img_grided)

        cv2.destroyAllWindows() 
    
    def save_patches(self, img_path):
        patch_size_on_original = int(self.base_patch_size * self.patch_size_multiplier)
        if patch_size_on_original <= 0:
            print(f"Error: Invalid patch size ({patch_size_on_original}). Cannot save patches.")
            return

        img = cv2.imread(img_path)
        if img is None:
            print(f"Error: Could not reload image for saving patches: {img_path}")
            return
        
        height, width = img.shape[:2]
        patches_saved_count = 0

        for y in range(self.patch_vertical_offset, height, patch_size_on_original):
            if y >= height or y + patch_size_on_original <= 0:
                continue 
            
            for x in range(self.patch_horizontal_offset, width, patch_size_on_original):
                if x >= width or x + patch_size_on_original <= 0:
                    continue
                
                if y < 0 or x < 0 or \
                   (y + patch_size_on_original > height) or \
                   (x + patch_size_on_original > width):
                    continue
                
                patch = img[y:y + patch_size_on_original, x:x + patch_size_on_original]
                
                image_name_base = os.path.basename(img_path)
                image_name_no_ext = os.path.splitext(image_name_base)[0]
                
                # Filename uses indices relative to a (0,0) aligned grid for simplicity.
                patch_y_idx = y // patch_size_on_original
                patch_x_idx = x // patch_size_on_original
                patch_filename = f"{image_name_no_ext}_patch_{patch_size_on_original}x{patch_size_on_original}_{patch_y_idx}_{patch_x_idx}_{self.patch_horizontal_offset}_{self.patch_vertical_offset}.png"
                
                cv2.imwrite(os.path.join(self.output_folder, patch_filename), patch)
                patches_saved_count += 1
        
        print(f"Saved {patches_saved_count} patches for {os.path.basename(img_path)}.")

In [8]:
classes = sorted(os.listdir(data_dir))
path = os.path.join(data_dir, classes[1])
gui = OpenCVGUI(path, target_dir)
gui.run()

Processing image: /home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/preprocessed/Cabomba/Cabomba 4.JPG
Image shape: (1920, 2560, 3)
Increasing patch size multiplier.
Increasing patch size multiplier.
Increasing patch size multiplier.
Increasing patch size multiplier.
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -10
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -20
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -30
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -40
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -50
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -60
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -70
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -80
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -90
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -100
Moving grid left.
Current offsets - Vertical: 0, Horizontal: -110